# NB-09-02 — CNN Classification Workbench

**Module M09 · Computer Vision** · [Lesson L09-03](../lessons/L09-03-image-classification-and-cnns.md)

Train the module's `TinyCNN` on two synthetic datasets — traffic signs (easy, colour-separated) and leaves
(harder, texture-separated) — and compare it against a classical baseline on exactly the same split. The
comparison is the point: a model that is only compared with itself has not been evaluated.

In [1]:
import sys
sys.path.insert(0, "../code")          # the module's reference code lives one directory up

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import vision_core as v

print("vision_core self-test:", v.run_tests())

vision_core self-test: True


## 1. Data and split

Fixed seed, stratified 75/25 split. Report the split sizes; a 30-image test set supports "no errors on
this sample", not "accurate".

In [2]:
images, labels, names = v.make_sign_images(n_per_class=40, size=32, seed=0)
cut = int(len(labels) * 0.75)
train_x, train_y = images[:cut], labels[:cut]
test_x, test_y = images[cut:], labels[cut:]
print("train:", train_x.shape, "test:", test_x.shape, "classes:", names)
print("class counts:", dict(zip(*np.unique(train_y, return_counts=True))))

train: (90, 32, 32, 3) test: (30, 32, 32, 3) classes: ['stop', 'warning', 'mandatory']
class counts: {np.int64(0): np.int64(30), np.int64(1): np.int64(32), np.int64(2): np.int64(28)}


## 2. Count the parameters before training

Conv layer parameters are $C_{out}(C_{in}k^2 + 1)$; the flatten size follows from two 2×2 pools on a
32×32 input.

In [3]:
print("conv1:", 8 * (3 * 3 * 3 + 1))
print("conv2:", 16 * (8 * 3 * 3 + 1))
print("flatten:", 16 * (32 // 4) * (32 // 4))

conv1: 224
conv2: 1168
flatten: 1024


## 3. Train and read the curve

Three epochs, default settings. Watch training accuracy: it should climb from near chance (1/3) to above
0.8.

In [4]:
model, history = v.train_classifier(train_x, train_y, epochs=3, seed=0)
for row in history:
    print(f"epoch {row['epoch']}: loss {row['loss']:.3f}  train accuracy {row['train_accuracy']:.3f}")

epoch 1: loss 1.076  train accuracy 0.444
epoch 2: loss 0.719  train accuracy 0.644
epoch 3: loss 0.249  train accuracy 0.878


## 4. Evaluate on held-out images

Accuracy plus a confusion matrix. Rows are truth, columns are prediction.

In [5]:
predicted = v.predict(model, test_x)
print("test accuracy:", round(float((predicted == test_y).mean()), 3))
print(v.confusion_matrix(predicted, test_y, 3))

test accuracy: 1.0
[[10  0  0]
 [ 0  8  0]
 [ 0  0 12]]


## 5. Augmentation as a prior

Flip, rotation and brightness jitter assume the label survives them. Compare the test accuracy with and
without. On this data the honest answer may be "no difference" — write that if it is true.

In [6]:
aug_model, aug_history = v.train_classifier(train_x, train_y, epochs=3, seed=0, use_augmentation=True)
print("training accuracy per epoch:", [round(h["train_accuracy"], 3) for h in aug_history])
print("augmented test accuracy:", round(float((v.predict(aug_model, test_x) == test_y).mean()), 3))

training accuracy per epoch: [0.356, 0.644, 1.0]
augmented test accuracy: 1.0


## 6. The classical baseline on the same split

HOG plus logistic regression, same training images, same test images.

In [7]:
from sklearn.linear_model import LogisticRegression

def hog_features(batch, cell=8):
    return np.stack([v.hog_descriptor(v.to_gray(img), cell=cell) for img in batch])

clf = LogisticRegression(max_iter=1000).fit(hog_features(train_x), train_y)
print("HOG + logistic regression test accuracy:", round(clf.score(hog_features(test_x), test_y), 3))

HOG + logistic regression test accuracy: 1.0


## 7. A harder dataset: leaves

Texture rather than colour, 60 images per class, 8 epochs. The training/test gap is the diagnostic.

In [8]:
leaf_x, leaf_y, leaf_names = v.make_leaf_images(n_per_class=60, size=32, seed=1)
cut = 140
leaf_model, leaf_history = v.train_classifier(leaf_x[:cut], leaf_y[:cut], epochs=8, seed=0)
print("training accuracy:", [round(h["train_accuracy"], 2) for h in leaf_history])
leaf_pred = v.predict(leaf_model, leaf_x[cut:])
print("test accuracy:", round(float((leaf_pred == leaf_y[cut:]).mean()), 3))
print(v.confusion_matrix(leaf_pred, leaf_y[cut:], 3))

leaf_clf = LogisticRegression(max_iter=1000).fit(hog_features(leaf_x[:cut]), leaf_y[:cut])
print("HOG baseline test accuracy:", round(leaf_clf.score(hog_features(leaf_x[cut:]), leaf_y[cut:]), 3))

training accuracy: [0.45, 0.51, 0.67, 0.68, 0.65, 0.67, 0.81, 0.76]
test accuracy: 0.95
[[12  2  0]
 [ 0 11  0]
 [ 0  0 15]]
HOG baseline test accuracy: 0.95


## 8. Exercise

1. Train the leaf model for 8 epochs with augmentation and report the difference.
2. Build a classifier that always predicts class 0; report its accuracy and per-class recall on the leaf
   test set. Which metric exposes it?
3. Explain in three sentences why the CNN is not automatically better than HOG here.

## Self-assessment

- What does a training accuracy of 0.76 with test accuracy of 0.95 tell you about the model and the data?
- Why is 30 test images insufficient for a deployment claim?
- Which change would you expect to help most on real field photographs: more depth, more data, or more
  augmentation? Justify with the evidence in this notebook.